In [1]:
import pandas as pd
import re
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.notebook import tqdm

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

tqdm.pandas()
print("Imports complete.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kunja\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kunja\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kunja\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kunja\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kunja\AppData\Roaming\nltk_data...


Imports complete.


In [2]:
# Load Data

df = pd.read_csv("comments_cleaned.csv")
print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)


Loaded 6,241 rows
Columns: ['video_id', 'video_title', 'author', 'comment_text', 'comment_published_at', 'comment_like_count', 'clean_comment', 'comment_length_chars', 'comment_length_words']


,video_id,video_title,author,comment_text,comment_published_at,comment_like_count,clean_comment,comment_length_chars,comment_length_words
0,laZpTO7IFtA,Is 67 just brain rot?,@ExoticNibbles2,Nothing makes me feel older then watching a 15...,2025-10-14T22:24:09Z,7902,nothing makes me feel older then watching a 15...,109,20
1,laZpTO7IFtA,Is 67 just brain rot?,@andrewbunnell7576,6 x 7 = 42 The answer to Life the Universe and...,2025-10-13T20:12:09Z,7692,6 x 7 = 42 the answer to life the universe and...,57,13
2,laZpTO7IFtA,Is 67 just brain rot?,@ColinPaddock,All I know is 6 is afraid of 7.,2025-10-14T06:39:54Z,6194,all i know is 6 is afraid of 7.,31,9


In [3]:
# Custom Stopwords and Preprocessing Function

CUSTOM_STOPWORDS = {
    # YouTube comment filler
    "video", "watch", "watching", "watched", "channel", "subscribe",
    "comment", "youtube", "like", "liked",
    # Discourse filler / intensifiers
    "just", "really", "actually", "basically", "literally", "honestly",
    "also", "even", "still", "already", "always", "never", "ever",
    "okay", "ok", "alright", "yeah", "yes", "nah", "nope", "yep",
    # Casual speech
    "gonna", "wanna", "gotta", "kinda", "sorta", "lol", "lmao",
    "haha", "hahaha", "omg", "wow",
    # High-frequency low-signal verbs/nouns
    "got", "get", "getting", "know", "knowing", "think", "thinking",
    "make", "making", "made", "say", "said", "saying", "see", "seen",
    "come", "coming", "came", "back", "right", "good", "bad", "great",
    "thing", "things", "way", "time", "lot", "many", "much",
    "people", "one", "two", "three", "something", "anything",
    "nothing", "everything", "someone", "anyone", "everyone",
    # Surviving contractions after tokenisation
    "im", "ive", "dont", "cant", "wont", "isnt", "wasnt",
    "didnt", "thats", "youre", "theyre", "weve", "wouldnt",
}

LEMMATIZE = True  # set to False to skip lemmatisation

lemmatizer = WordNetLemmatizer()
std_stopwords = set(stopwords.words("english"))
all_stopwords = std_stopwords | CUSTOM_STOPWORDS


def preprocess_for_lda(text: str) -> list:
    # Step 1: lowercase (clean_comment is already lowercase, but enforce)
    text = str(text).lower()

    # Step 2: remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Step 3: remove timestamps (e.g. 4:22, 1:05:30)
    text = re.sub(r"\b\d{1,2}:\d{2}(:\d{2})?\b", "", text)

    # Step 4: remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Step 5: remove standalone numbers
    text = re.sub(r"\b\d+\b", "", text)

    # Step 6: tokenise
    tokens = word_tokenize(text)

    # Step 7 + 8: remove standard + custom stopwords
    tokens = [t for t in tokens if t not in all_stopwords]

    # Step 9: remove short tokens (< 3 characters)
    tokens = [t for t in tokens if len(t) >= 3]

    # Step 10 (optional): lemmatise
    if LEMMATIZE:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens


print(f"Lemmatisation: {'ON' if LEMMATIZE else 'OFF'}")
print(f"Total stopwords (standard + custom): {len(all_stopwords):,}")


Lemmatisation: ON
Total stopwords (standard + custom): 293


In [4]:
# Preprocessing and Filter Short Comments

df["tokens"] = df["clean_comment"].progress_apply(preprocess_for_lda)

# Step 11: remove comments with fewer than 5 meaningful tokens
before = len(df)
df = df[df["tokens"].apply(len) >= 5].reset_index(drop=True)
after = len(df)

print(f"Removed {before - after:,} comments (fewer than 5 tokens after preprocessing).")
print(f"Remaining: {after:,} comments")

# Join tokens back into a string for CountVectorizer
df["processed_text"] = df["tokens"].apply(lambda toks: " ".join(toks))


  0%|          | 0/6241 [00:00<?, ?it/s]

Removed 2,079 comments (fewer than 5 tokens after preprocessing).
Remaining: 4,162 comments


In [5]:
# Sanity Check: Most Common Tokens

all_tokens = [tok for toks in df["tokens"] for tok in toks]
token_freq = Counter(all_tokens)

print(f"Total tokens in corpus: {len(all_tokens):,}")
print(f"Unique tokens: {len(token_freq):,}")
print(f"\nTop 40 most frequent tokens:")
for word, count in token_freq.most_common(40):
    print(f"  {word:<20} {count:>6}")

# Review this output before continuing. If you see obvious noise words in the top 40, add them to CUSTOM_STOPWORDS in Cell 3 and re-run from there.

Total tokens in corpus: 74,853
Unique tokens: 11,505

Top 40 most frequent tokens:
  kid                     603
  brain                   491
  phone                   487
  year                    459
  gen                     448
  social                  446
  medium                  423
  feel                    397
  day                     394
  content                 329
  word                    314
  use                     312
  old                     302
  would                   300
  short                   290
  used                    278
  video                   272
  slang                   265
  brainrot                260
  thank                   257
  parent                  256
  love                    244
  language                240
  internet                238
  life                    237
  rot                     236
  need                    235
  work                    232
  attention               231
  new                     228
  want           

In [6]:
# CountVectorizer

# Parameters to tune when running LDA:
#   min_df     - ignore terms in fewer than N documents (filters rare noise)
#   max_df     - ignore terms in more than X% of documents (filters ubiquitous terms)
#   max_features - vocabulary cap (smaller = faster LDA, larger = more granular topics)

vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.90,
    max_features=2000,
)

dtm = vectorizer.fit_transform(df["processed_text"])
vocab = vectorizer.get_feature_names_out()

print(f"Document-Term Matrix shape: {dtm.shape}")
print(f"  {dtm.shape[0]:,} documents  x  {dtm.shape[1]:,} terms")
print(f"\nSample vocabulary (first 30 terms): {list(vocab[:30])}")


Document-Term Matrix shape: (4162, 2000)
  4,162 documents  x  2,000 terms

Sample vocabulary (first 30 terms): ['1960s', '1st', '2000s', '2010s', '30', '60', '70', '80', '90', 'ability', 'able', 'absolute', 'absolutely', 'absorb', 'academic', 'accent', 'accept', 'access', 'accessible', 'account', 'accurate', 'achieve', 'acronym', 'across', 'act', 'acting', 'action', 'active', 'actively', 'activity']


In [7]:
# Save output

output_cols = [
    c for c in ["video_id", "video_title", "author",
                 "comment_published_at", "comment_like_count"]
    if c in df.columns
]
output_cols.append("processed_text")

df[output_cols].to_csv("topic_modelling_preprocess.csv", index=False, encoding="utf-8")

print(f"Saved {len(df):,} rows to topic_modelling_preprocess.csv")
print(f"Columns: {output_cols}")
df[output_cols].head(3)


Saved 4,162 rows to topic_modelling_preprocess.csv
Columns: ['video_id', 'video_title', 'author', 'comment_published_at', 'comment_like_count', 'processed_text']


,video_id,video_title,author,comment_published_at,comment_like_count,processed_text
0,laZpTO7IFtA,Is 67 just brain rot?,@ExoticNibbles2,2025-10-14T22:24:09Z,7902,make feel older minute breaking slang young ki...
1,laZpTO7IFtA,Is 67 just brain rot?,@codahighland,2025-12-10T16:03:28Z,121,completely context thought solo version
2,laZpTO7IFtA,Is 67 just brain rot?,@yoberry.,2025-10-18T07:29:08Z,2084,next kid say going punish
